<a href="https://colab.research.google.com/github/sayanm1/LTIMindtree-GenAI-Projects/blob/main/Project_1_Enterprise_RAG_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Force compatible versions to satisfy both ChromaDB and Colab's internal environment
!pip install -q "opentelemetry-api==1.42.1" "opentelemetry-sdk==1.42.1"
!pip install -q transformers torch chromadb sentence-transformers accelerate

In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# =====================================================================
# 1. ENTERPRISE DATA INGESTION & VECTOR STORAGE
# =====================================================================
print("Initializing Cloud Vector Database (ChromaDB)...")

corporate_documents = [
    "LTIMindtree Remote Work Policy: Employees are permitted to work remotely up to 2 days per week, subject to managerial approval and project delivery requirements.",
    "LTIMindtree Security Guidelines: Multi-Factor Authentication (MFA) must be enabled on all corporate laptops. Accessing client databases over public Wi-Fi without a secure VPN is strictly prohibited.",
    "LTIMindtree Innovation Lab: The GenAI center of excellence focuses on scaling Small Language Models (SLMs) and implementing highly efficient enterprise Retrieval-Augmented Generation systems."
]

doc_ids = [f"id_{i}" for i in range(len(corporate_documents))]
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedding_model.encode(corporate_documents).tolist()

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="corporate_knowledge_base")

collection.add(
    embeddings=doc_embeddings,
    documents=corporate_documents,
    ids=doc_ids
)
print("✅ Vector Database successfully initialized.")

# =====================================================================
# 2. RETRIEVAL WITH THRESHOLD GUARDRAIL
# =====================================================================
def retrieve_relevant_context(user_query):
    """
    Queries vector DB. Returns the context text ONLY if it's high confidence.
    Otherwise returns None to signify a general query.
    """
    query_vector = embedding_model.encode([user_query]).tolist()
    results = collection.query(query_embeddings=query_vector, n_results=1)

    # ChromaDB distance score (Lower distance means higher context similarity)
    distance_score = results['distances'][0][0]

    # Threshold 1.2 isolates documents unrelated to our LTIMindtree text matrix
    if distance_score < 1.2:
        return results['documents'][0][0]
    return None

# =====================================================================
# 3. GENERATION LAYER (LOADING THE MODEL VIA THE CLOUD GPU)
# =====================================================================
print("\nLoading Optimized Text Generation Model...")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("✅ Large Language Model loaded onto cloud GPU.")

# =====================================================================
# 4. INTELLECTUAL EXECUTION ROUTER
# =====================================================================
def ask_enterprise_ai(question):
    """Routes the query: Uses database context if matched, otherwise defaults to general LLM logic."""
    context = retrieve_relevant_context(question)

    if context:
        # Scenario A: High-confidence database match found
        mode = "Contextual RAG Mode"
        prompt = f"""
        You are an expert corporate AI assistant. Use the verified corporate context below to answer the question directly.

        Verified Context: {context}
        Question: {question}

        Answer:"""
    else:
        # Scenario B: Fallback mode for general knowledge questions
        mode = "General Intelligence Mode"
        prompt = f"""
        You are an expert technical assistant. Answer the user's question comprehensively based on industry best practices.

        Question: {question}

        Answer:"""

    # Execute generation pipeline
    with torch.no_grad():
        output = generator(prompt, max_new_tokens=250, temperature=0.5, do_sample=True)

    generated_text = output[0]['generated_text']
    response = generated_text.split("Answer:")[-1].strip()

    print(f"\n--- Enterprise AI Session [{mode}] ---")
    print(f"User Question: {question}")
    if context:
        print(f"Retrieved Vector Context: {context}")
    print(f"AI Response:\n{response}")
    print(f"---------------------------------------------------\n")

# =====================================================================
# 5. RUNTIME INTERACTIVE LOOP
# =====================================================================
print("\n🚀 Enterprise AI Assistant is active!")
print("Type your question below. Type 'exit' to stop the session.\n")

while True:
    user_input = input("Enter your question: ").strip()
    if user_input.lower() in ['exit', 'quit', 'q']:
        print("Closing secure session. Goodbye!")
        break
    if not user_input:
        continue
    ask_enterprise_ai(user_input)